# Project 8: Quantum Reinforcement Learning

## Introduction

Reinforcement Learning (RL) is a paradigm of machine learning where an **agent** interacts with an **environment** to maximize cumulative **reward**. The agent observes a **state** $s$, takes an **action** $a$, receives a reward $r$, and transitions to a new state $s'$.

### Core RL Concepts

| Concept | Description |
|---------|-------------|
| **State** $s \in \mathcal{S}$ | Current observation of the environment |
| **Action** $a \in \mathcal{A}$ | Decision taken by the agent |
| **Reward** $r(s, a)$ | Scalar feedback from the environment |
| **Policy** $\pi(a|s)$ | Probability of taking action $a$ in state $s$ |
| **Value** $V^\pi(s)$ | Expected cumulative reward from state $s$ under policy $\pi$ |
| **Q-value** $Q^\pi(s,a)$ | Expected cumulative reward for action $a$ in state $s$ |

### Quantum Policy Representation

A **quantum policy** uses a parameterized quantum circuit $U(\theta, s)$ to map states to action probabilities:

$$\pi_\theta(a|s) = |\langle a | U(\theta, s) | 0 \rangle|^2$$

where $\theta$ are trainable parameters, $s$ is encoded into the circuit, and measurement outcomes correspond to actions.

This project compares three approaches on a FrozenLake navigation task:
1. **Classical Tabular Q-Learning**
2. **Quantum REINFORCE** (policy gradient)
3. **Hybrid Quantum-Classical DQN**

In [ ]:
# --- Imports ---
import pennylane as qml
from pennylane import numpy as pnp
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import deque
import random
import warnings
warnings.filterwarnings('ignore')

# Reproducibility
np.random.seed(42)
torch.manual_seed(42)
random.seed(42)

print("PennyLane version:", qml.__version__)
print("PyTorch version:", torch.__version__)

## Theoretical Background

### Q-Learning Update Rule

The classical Q-learning algorithm updates the Q-table via the Bellman equation:

$$Q(s, a) \leftarrow Q(s, a) + \alpha \Big[ r + \gamma \max_{a'} Q(s', a') - Q(s, a) \Big]$$

where $\alpha$ is the learning rate and $\gamma$ is the discount factor.

### Quantum Policy Gradient (REINFORCE)

The policy gradient theorem gives us the gradient of the expected return:

$$\nabla_\theta J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta} \left[ \sum_{t=0}^{T} \nabla_\theta \log \pi_\theta(a_t|s_t) \cdot G_t \right]$$

where $G_t = \sum_{k=t}^{T} \gamma^{k-t} r_k$ is the return from time step $t$.

### Variational Quantum Policy

A variational quantum policy circuit consists of:
1. **State encoding**: Map classical state $s$ into qubit rotations, e.g., $R_Y(s_i)$
2. **Variational layers**: Trainable entangling gates with parameters $\theta$
3. **Measurement**: Read out action probabilities from qubit measurement statistics

The circuit acts as a universal function approximator in the space of quantum states, potentially offering advantages in expressibility and trainability for certain RL tasks.

In [ ]:
# --- Custom 4x4 FrozenLake Grid Environment ---

class FrozenLakeEnv:
    """
    Custom 4x4 FrozenLake environment.
    
    Grid layout:
      S  F  F  F
      F  H  F  H
      F  F  F  H
      H  F  F  G
    
    S = Start (state 0), F = Frozen (safe tile)
    H = Hole (terminal, reward=-1), G = Goal (terminal, reward=+1)
    Actions: 0=Left, 1=Down, 2=Right, 3=Up
    """
    
    GRID = [
        ['S', 'F', 'F', 'F'],
        ['F', 'H', 'F', 'H'],
        ['F', 'F', 'F', 'H'],
        ['H', 'F', 'F', 'G']
    ]
    
    N_STATES = 16
    N_ACTIONS = 4
    ACTION_NAMES = ['Left', 'Down', 'Right', 'Up']
    ACTION_ARROWS = ['\u2190', '\u2193', '\u2192', '\u2191']
    
    def __init__(self, slip_prob=0.1):
        self.slip_prob = slip_prob
        self.state = 0
        
    def reset(self):
        self.state = 0
        return self.state
    
    def _pos(self, state):
        return state // 4, state % 4
    
    def _state_from_pos(self, row, col):
        return row * 4 + col
    
    def _cell_type(self, state):
        r, c = self._pos(state)
        return self.GRID[r][c]
    
    def step(self, action):
        # With small probability, slip to a random action
        if np.random.random() < self.slip_prob:
            action = np.random.randint(0, 4)
        
        r, c = self._pos(self.state)
        
        if action == 0:    # Left
            c = max(0, c - 1)
        elif action == 1:  # Down
            r = min(3, r + 1)
        elif action == 2:  # Right
            c = min(3, c + 1)
        elif action == 3:  # Up
            r = max(0, r - 1)
        
        self.state = self._state_from_pos(r, c)
        cell = self._cell_type(self.state)
        
        if cell == 'H':
            return self.state, -1.0, True
        elif cell == 'G':
            return self.state, 1.0, True
        else:
            return self.state, -0.01, False  # small step penalty


def visualize_grid(env=None):
    """Visualize the FrozenLake grid layout."""
    grid = FrozenLakeEnv.GRID
    fig, ax = plt.subplots(1, 1, figsize=(5, 5))
    
    color_map = {'S': '#90EE90', 'F': '#ADD8E6', 'H': '#FFB6C1', 'G': '#FFD700'}
    
    for i in range(4):
        for j in range(4):
            cell = grid[i][j]
            rect = plt.Rectangle((j, 3 - i), 1, 1, facecolor=color_map[cell],
                                  edgecolor='black', linewidth=2)
            ax.add_patch(rect)
            label = {'S': 'Start', 'F': 'Frozen', 'H': 'Hole', 'G': 'Goal'}[cell]
            ax.text(j + 0.5, 3 - i + 0.6, label, ha='center', va='center',
                    fontsize=10, fontweight='bold')
            ax.text(j + 0.5, 3 - i + 0.3, f's={i*4+j}', ha='center', va='center',
                    fontsize=8, color='gray')
    
    ax.set_xlim(0, 4)
    ax.set_ylim(0, 4)
    ax.set_aspect('equal')
    ax.set_title('FrozenLake 4x4 Grid', fontsize=14, fontweight='bold')
    ax.axis('off')
    
    patches = [mpatches.Patch(color=c, label=l) for l, c in
               [('Start', '#90EE90'), ('Frozen', '#ADD8E6'),
                ('Hole', '#FFB6C1'), ('Goal', '#FFD700')]]
    ax.legend(handles=patches, loc='upper left', bbox_to_anchor=(1, 1))
    plt.tight_layout()
    plt.show()

visualize_grid()

In [ ]:
# --- Classical Tabular Q-Learning ---

def train_q_learning(n_episodes=1000, alpha=0.1, gamma=0.99, epsilon_start=1.0,
                     epsilon_end=0.01, epsilon_decay=0.995, max_steps=50):
    """
    Train a tabular Q-learning agent on FrozenLake.
    Returns: Q-table (16x4), episode reward history
    """
    env = FrozenLakeEnv(slip_prob=0.05)
    Q = np.zeros((env.N_STATES, env.N_ACTIONS))
    episode_rewards = []
    epsilon = epsilon_start
    
    for ep in range(n_episodes):
        state = env.reset()
        total_reward = 0
        
        for step in range(max_steps):
            # Epsilon-greedy action selection
            if np.random.random() < epsilon:
                action = np.random.randint(env.N_ACTIONS)
            else:
                action = np.argmax(Q[state])
            
            next_state, reward, done = env.step(action)
            
            # Q-learning update (off-policy TD)
            best_next = np.max(Q[next_state]) if not done else 0
            Q[state, action] += alpha * (reward + gamma * best_next - Q[state, action])
            
            state = next_state
            total_reward += reward
            
            if done:
                break
        
        episode_rewards.append(total_reward)
        epsilon = max(epsilon_end, epsilon * epsilon_decay)
    
    return Q, episode_rewards


# Train the Q-learning agent
print("Training Classical Q-Learning (1000 episodes)...")
Q_table, q_rewards = train_q_learning(n_episodes=1000)
print(f"Final 100-episode average reward: {np.mean(q_rewards[-100:]):.3f}")

# Plot training curve and learned policy
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Reward curve with moving average
window = 50
moving_avg = np.convolve(q_rewards, np.ones(window)/window, mode='valid')
axes[0].plot(q_rewards, alpha=0.3, color='blue', label='Episode reward')
axes[0].plot(range(window-1, len(q_rewards)), moving_avg, color='red',
             linewidth=2, label=f'{window}-ep moving avg')
axes[0].set_xlabel('Episode')
axes[0].set_ylabel('Total Reward')
axes[0].set_title('Q-Learning Training Curve')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Policy arrows on grid
grid = FrozenLakeEnv.GRID
color_map = {'S': '#90EE90', 'F': '#ADD8E6', 'H': '#FFB6C1', 'G': '#FFD700'}
arrow_dx = [-0.25, 0, 0.25, 0]   # Left, Down, Right, Up
arrow_dy = [0, -0.25, 0, 0.25]

for i in range(4):
    for j in range(4):
        cell = grid[i][j]
        rect = plt.Rectangle((j, 3 - i), 1, 1, facecolor=color_map[cell],
                              edgecolor='black', linewidth=2)
        axes[1].add_patch(rect)
        
        s = i * 4 + j
        if cell in ('H', 'G'):
            label = 'Hole' if cell == 'H' else 'Goal'
            axes[1].text(j + 0.5, 3 - i + 0.5, label, ha='center', va='center',
                         fontsize=9, fontweight='bold')
        else:
            best_a = np.argmax(Q_table[s])
            axes[1].annotate('',
                xy=(j + 0.5 + arrow_dx[best_a], 3 - i + 0.5 + arrow_dy[best_a]),
                xytext=(j + 0.5, 3 - i + 0.5),
                arrowprops=dict(arrowstyle='->', lw=2, color='darkblue'))

axes[1].set_xlim(0, 4)
axes[1].set_ylim(0, 4)
axes[1].set_aspect('equal')
axes[1].set_title('Learned Q-Learning Policy', fontsize=13, fontweight='bold')
axes[1].axis('off')
plt.tight_layout()
plt.show()

## Quantum Policy Network

We now construct a **parameterized quantum circuit (PQC)** to serve as a policy network. The quantum policy maps a discrete state to a probability distribution over actions.

**Architecture:**
- **Input encoding**: The state index $s$ is converted to a 4-bit binary representation $(b_3, b_2, b_1, b_0)$. Each bit is encoded as an $R_Y$ rotation on the corresponding qubit.
- **Variational layers**: Multiple layers of single-qubit rotations ($R_Y$, $R_Z$) and entangling CNOT gates.
- **Measurement**: Qubits 0 and 1 are measured. The four computational basis states $|00\rangle, |01\rangle, |10\rangle, |11\rangle$ correspond to the four actions (Left, Down, Right, Up).

The output probabilities naturally satisfy $\sum_a \pi_\theta(a|s) = 1$.

In [ ]:
# --- Parameterized Quantum Policy Circuit ---

n_qubits = 4
n_layers = 3
dev_policy = qml.device('default.qubit', wires=n_qubits)

def state_to_angles(state_idx):
    """Convert state index (0-15) to 4 rotation angles via binary encoding."""
    bits = [(state_idx >> i) & 1 for i in range(n_qubits)]
    return [b * np.pi for b in bits]


@qml.qnode(dev_policy, interface='torch', diff_method='backprop')
def quantum_policy_circuit(params, state_angles):
    """
    Quantum policy circuit.
    
    Args:
        params: shape (n_layers, n_qubits, 2) - trainable parameters
        state_angles: shape (n_qubits,) - state encoding angles
    
    Returns:
        Probabilities of measurement on qubits 0 and 1 (4 outcomes = 4 actions)
    """
    # State encoding via RY rotations
    for i in range(n_qubits):
        qml.RY(state_angles[i], wires=i)
    
    # Variational layers
    for layer in range(n_layers):
        # Single-qubit rotations
        for i in range(n_qubits):
            qml.RY(params[layer, i, 0], wires=i)
            qml.RZ(params[layer, i, 1], wires=i)
        # Entangling CNOT ring
        for i in range(n_qubits):
            qml.CNOT(wires=[i, (i + 1) % n_qubits])
    
    # Measure qubits 0 and 1 to get 4 action probabilities
    return qml.probs(wires=[0, 1])


# Test the circuit
test_params = torch.randn(n_layers, n_qubits, 2, dtype=torch.float64) * 0.1
test_angles = torch.tensor(state_to_angles(0), dtype=torch.float64)
probs = quantum_policy_circuit(test_params, test_angles)
print("Action probabilities for state 0:", probs.detach().numpy())
print("Sum of probabilities:", probs.sum().item())

# Draw the circuit
print("\nQuantum Policy Circuit:")
print(qml.draw(quantum_policy_circuit, decimals=2)(test_params, test_angles))

In [ ]:
# --- Quantum REINFORCE Algorithm ---

def train_quantum_reinforce(n_episodes=500, gamma=0.99, lr=0.01, max_steps=50):
    """
    Train a quantum policy using the REINFORCE algorithm.
    
    The quantum circuit outputs action probabilities directly.
    We use the policy gradient theorem to update circuit parameters.
    """
    env = FrozenLakeEnv(slip_prob=0.05)
    
    # Initialize variational parameters
    params = torch.randn(n_layers, n_qubits, 2, dtype=torch.float64,
                         requires_grad=True) * 0.5
    optimizer = optim.Adam([params], lr=lr)
    
    episode_rewards = []
    
    for ep in range(n_episodes):
        state = env.reset()
        log_probs = []
        rewards = []
        
        for step in range(max_steps):
            # Get action probabilities from quantum circuit
            angles = torch.tensor(state_to_angles(state), dtype=torch.float64)
            action_probs = quantum_policy_circuit(params, angles)
            
            # Add small epsilon for numerical stability
            action_probs = action_probs + 1e-8
            action_probs = action_probs / action_probs.sum()
            
            # Sample action from the quantum policy distribution
            dist = torch.distributions.Categorical(action_probs)
            action = dist.sample()
            log_probs.append(dist.log_prob(action))
            
            # Take step in environment
            next_state, reward, done = env.step(action.item())
            rewards.append(reward)
            state = next_state
            
            if done:
                break
        
        # Compute discounted returns
        returns = []
        G = 0
        for r in reversed(rewards):
            G = r + gamma * G
            returns.insert(0, G)
        returns = torch.tensor(returns, dtype=torch.float64)
        
        # Normalize returns for training stability
        if len(returns) > 1:
            returns = (returns - returns.mean()) / (returns.std() + 1e-8)
        
        # Policy gradient loss: -E[log pi * G]
        loss = 0
        for lp, G_t in zip(log_probs, returns):
            loss -= lp * G_t
        
        # Backpropagation through the quantum circuit
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_reward = sum(rewards)
        episode_rewards.append(total_reward)
        
        if (ep + 1) % 100 == 0:
            avg = np.mean(episode_rewards[-100:])
            print(f"Episode {ep+1}/{n_episodes} | Avg reward (last 100): {avg:.3f}")
    
    return params.detach(), episode_rewards


print("Training Quantum REINFORCE (500 episodes)...")
q_policy_params, qr_rewards = train_quantum_reinforce(n_episodes=500)
print(f"\nFinal 100-episode average reward: {np.mean(qr_rewards[-100:]):.3f}")

## Hybrid Quantum-Classical DQN

We combine a classical neural network encoder with a quantum processing layer to form a **Hybrid DQN**:

1. **Classical encoder**: Maps the state one-hot vector (dim 16) to rotation angles (dim 4) for the quantum circuit
2. **Quantum layer**: A variational quantum circuit processes the encoded state and outputs Pauli-Z expectation values
3. **Classical output**: Maps quantum measurement outcomes to Q-values for each action

Training uses:
- **Epsilon-greedy** exploration with annealing
- **Experience replay** buffer for sample efficiency
- **Target network** updated periodically for training stability

In [ ]:
# --- Hybrid Quantum-Classical DQN ---

n_qubits_dqn = 4
n_layers_dqn = 2
dev_dqn = qml.device('default.qubit', wires=n_qubits_dqn)


@qml.qnode(dev_dqn, interface='torch', diff_method='backprop')
def quantum_layer(inputs, weights):
    """Quantum processing layer for the hybrid DQN."""
    # Encode classical features as rotations
    for i in range(n_qubits_dqn):
        qml.RY(inputs[i], wires=i)
    
    # Variational layers with entanglement
    for layer in range(n_layers_dqn):
        for i in range(n_qubits_dqn):
            qml.RY(weights[layer, i, 0], wires=i)
            qml.RZ(weights[layer, i, 1], wires=i)
        for i in range(n_qubits_dqn):
            qml.CNOT(wires=[i, (i + 1) % n_qubits_dqn])
    
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits_dqn)]


class HybridDQN(nn.Module):
    """Hybrid quantum-classical Deep Q-Network."""
    
    def __init__(self):
        super().__init__()
        # Classical encoder: state one-hot (16) -> circuit angles (4)
        self.encoder = nn.Sequential(
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Linear(8, n_qubits_dqn),
            nn.Tanh()  # Output in [-1, 1], scaled to [-pi, pi]
        )
        
        # Quantum layer trainable parameters
        self.q_weights = nn.Parameter(
            torch.randn(n_layers_dqn, n_qubits_dqn, 2) * 0.5
        )
        
        # Classical decoder: quantum expectations (4) -> Q-values (4)
        self.output_layer = nn.Sequential(
            nn.Linear(n_qubits_dqn, 4)
        )
    
    def forward(self, state_onehot):
        # Classical encoding
        angles = self.encoder(state_onehot) * np.pi
        
        # Quantum processing
        q_out = quantum_layer(angles, self.q_weights)
        q_out = torch.stack(q_out) if isinstance(q_out, list) else q_out
        
        # Classical output head
        q_values = self.output_layer(q_out)
        return q_values


def state_to_onehot(state, n_states=16):
    """Convert state index to one-hot vector."""
    oh = torch.zeros(n_states, dtype=torch.float32)
    oh[state] = 1.0
    return oh


def train_hybrid_dqn(n_episodes=500, gamma=0.99, lr=0.005, epsilon_start=1.0,
                     epsilon_end=0.05, epsilon_decay=0.995, max_steps=50,
                     batch_size=32, buffer_size=2000, target_update=20):
    """
    Train the Hybrid DQN with experience replay and target network.
    """
    env = FrozenLakeEnv(slip_prob=0.05)
    
    policy_net = HybridDQN()
    target_net = HybridDQN()
    target_net.load_state_dict(policy_net.state_dict())
    target_net.eval()
    
    optimizer = optim.Adam(policy_net.parameters(), lr=lr)
    replay_buffer = deque(maxlen=buffer_size)
    episode_rewards = []
    epsilon = epsilon_start
    
    for ep in range(n_episodes):
        state = env.reset()
        total_reward = 0
        
        for step in range(max_steps):
            state_oh = state_to_onehot(state)
            
            # Epsilon-greedy action selection
            if np.random.random() < epsilon:
                action = np.random.randint(4)
            else:
                with torch.no_grad():
                    q_vals = policy_net(state_oh)
                    action = q_vals.argmax().item()
            
            next_state, reward, done = env.step(action)
            replay_buffer.append((state, action, reward, next_state, done))
            
            state = next_state
            total_reward += reward
            
            # Train from replay buffer when enough samples
            if len(replay_buffer) >= batch_size:
                batch = random.sample(replay_buffer, batch_size)
                
                loss = torch.tensor(0.0, dtype=torch.float32)
                for s, a, r, ns, d in batch:
                    s_oh = state_to_onehot(s)
                    ns_oh = state_to_onehot(ns)
                    
                    q_val = policy_net(s_oh)[a]
                    with torch.no_grad():
                        next_q = 0 if d else target_net(ns_oh).max().item()
                    target = r + gamma * next_q
                    loss += (q_val - target) ** 2
                
                loss /= batch_size
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            
            if done:
                break
        
        episode_rewards.append(total_reward)
        epsilon = max(epsilon_end, epsilon * epsilon_decay)
        
        # Periodically update target network
        if (ep + 1) % target_update == 0:
            target_net.load_state_dict(policy_net.state_dict())
        
        if (ep + 1) % 100 == 0:
            avg = np.mean(episode_rewards[-100:])
            print(f"Episode {ep+1}/{n_episodes} | Avg reward (last 100): {avg:.3f}")
    
    return policy_net, episode_rewards


print("Training Hybrid DQN (500 episodes)...")
hybrid_model, hdqn_rewards = train_hybrid_dqn(n_episodes=500)
print(f"\nFinal 100-episode average reward: {np.mean(hdqn_rewards[-100:]):.3f}")

In [ ]:
# --- Training Comparison Plots ---

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
window = 50

# --- Smoothed reward curves for all three approaches ---
for rewards, label, color in [
    (q_rewards, 'Classical Q-Learning (1000 ep)', 'blue'),
    (qr_rewards, 'Quantum REINFORCE (500 ep)', 'green'),
    (hdqn_rewards, 'Hybrid DQN (500 ep)', 'red')
]:
    ma = np.convolve(rewards, np.ones(window)/window, mode='valid')
    axes[0].plot(range(window-1, len(rewards)), ma, label=label,
                 color=color, linewidth=2)

axes[0].set_xlabel('Episode', fontsize=12)
axes[0].set_ylabel('Reward (Moving Avg)', fontsize=12)
axes[0].set_title('Training Comparison: Reward Curves', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# --- Final performance bar chart ---
final_avgs = [
    np.mean(q_rewards[-100:]),
    np.mean(qr_rewards[-100:]),
    np.mean(hdqn_rewards[-100:])
]
names = ['Q-Learning', 'Quantum\nREINFORCE', 'Hybrid\nDQN']
colors_bar = ['#4169E1', '#2E8B57', '#DC143C']
bars = axes[1].bar(names, final_avgs, color=colors_bar, edgecolor='black', width=0.5)
for bar, val in zip(bars, final_avgs):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{val:.3f}', ha='center', va='bottom', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Avg Reward (last 100 ep)', fontsize=12)
axes[1].set_title('Final Performance Comparison', fontsize=13, fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

In [ ]:
# --- Policy Visualization: Grid with Arrows + Value Heatmaps ---

def get_quantum_reinforce_policy(params):
    """Extract policy from trained quantum REINFORCE parameters."""
    policy = np.zeros((16, 4))
    for s in range(16):
        angles = torch.tensor(state_to_angles(s), dtype=torch.float64)
        probs = quantum_policy_circuit(params, angles).detach().numpy()
        policy[s] = probs
    return policy


def get_hybrid_dqn_policy(model):
    """Extract Q-values from trained Hybrid DQN."""
    policy = np.zeros((16, 4))
    model.eval()
    for s in range(16):
        with torch.no_grad():
            q_vals = model(state_to_onehot(s)).numpy()
            policy[s] = q_vals
    return policy


def plot_policy_and_values(policy_values, title, ax_policy, ax_value):
    """
    Plot policy arrows and value heatmap side by side.
    policy_values: (16, 4) array of Q-values or action probabilities
    """
    grid = FrozenLakeEnv.GRID
    color_map = {'S': '#90EE90', 'F': '#ADD8E6', 'H': '#FFB6C1', 'G': '#FFD700'}
    arrow_dx = [-0.25, 0, 0.25, 0]
    arrow_dy = [0, -0.25, 0, 0.25]
    
    # Policy arrows
    for i in range(4):
        for j in range(4):
            cell = grid[i][j]
            rect = plt.Rectangle((j, 3-i), 1, 1, facecolor=color_map[cell],
                                  edgecolor='black', linewidth=1.5)
            ax_policy.add_patch(rect)
            s = i * 4 + j
            if cell in ('H', 'G'):
                label = 'H' if cell == 'H' else 'G'
                ax_policy.text(j+0.5, 3-i+0.5, label, ha='center', va='center',
                              fontsize=14, fontweight='bold')
            else:
                best_a = np.argmax(policy_values[s])
                ax_policy.annotate('',
                    xy=(j+0.5+arrow_dx[best_a], 3-i+0.5+arrow_dy[best_a]),
                    xytext=(j+0.5, 3-i+0.5),
                    arrowprops=dict(arrowstyle='->', lw=2, color='darkblue'))
    
    ax_policy.set_xlim(0, 4)
    ax_policy.set_ylim(0, 4)
    ax_policy.set_aspect('equal')
    ax_policy.set_title(f'{title}\nPolicy', fontsize=11, fontweight='bold')
    ax_policy.axis('off')
    
    # Value heatmap (max over actions)
    values = np.max(policy_values, axis=1).reshape(4, 4)
    im = ax_value.imshow(values, cmap='RdYlGn', aspect='equal')
    for i in range(4):
        for j in range(4):
            ax_value.text(j, i, f'{values[i,j]:.2f}', ha='center', va='center',
                         fontsize=9, fontweight='bold')
    ax_value.set_title(f'{title}\nValue Heatmap', fontsize=11, fontweight='bold')
    ax_value.set_xticks([])
    ax_value.set_yticks([])
    plt.colorbar(im, ax=ax_value, shrink=0.8)


# Extract policies from all trained models
qr_policy = get_quantum_reinforce_policy(q_policy_params)
hdqn_policy = get_hybrid_dqn_policy(hybrid_model)

# Create 3-row visualization
fig, axes = plt.subplots(3, 2, figsize=(10, 14))

plot_policy_and_values(Q_table, 'Classical Q-Learning', axes[0, 0], axes[0, 1])
plot_policy_and_values(qr_policy, 'Quantum REINFORCE', axes[1, 0], axes[1, 1])
plot_policy_and_values(hdqn_policy, 'Hybrid DQN', axes[2, 0], axes[2, 1])

plt.suptitle('Policy Comparison Across All Approaches', fontsize=15,
             fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## Conclusion

### Summary of Results

| Approach | Method Type | Parameters | Key Advantage | Limitation |
|----------|------------|------------|---------------|------------|
| **Tabular Q-Learning** | Classical, value-based | 64 (Q-table) | Simple, guaranteed convergence | Does not scale to large state spaces |
| **Quantum REINFORCE** | Quantum, policy-based | 24 (circuit params) | Compact quantum policy, natural probability output | High variance gradients, slow convergence |
| **Hybrid DQN** | Hybrid, value-based | ~200 (mixed) | Combines classical feature extraction with quantum processing | More complex architecture, harder to train |

### Key Observations

1. **Classical Q-Learning** provides a strong baseline on this small discrete problem. For 16 states and 4 actions, the tabular approach is efficient and reliable.

2. **Quantum REINFORCE** demonstrates that parameterized quantum circuits can learn meaningful policies through gradient-based optimization. The Born-rule probabilities provide a natural softmax over actions without requiring additional normalization.

3. **Hybrid DQN** shows that quantum layers can be seamlessly integrated into deep RL architectures. The classical encoder handles feature extraction while the quantum circuit provides an expressive function approximator.

4. Quantum advantages may become more apparent with larger state/action spaces where the exponential Hilbert space dimension provides richer representational capacity.

### References

1. Jerbi, S., et al. "Parametrized quantum policies for reinforcement learning." *NeurIPS* (2021).
2. Skolik, A., et al. "Quantum agents in the Gym: a variational quantum algorithm for deep Q-learning." *Quantum* 6, 720 (2022).
3. Chen, S.Y.-C., et al. "Variational Quantum Circuits for Deep Reinforcement Learning." *IEEE Access* 8 (2020).
4. Lockwood, O., & Si, M. "Reinforcement Learning with Quantum Variational Circuits." *AIIDE* (2020).
5. Sutton, R.S. & Barto, A.G. *Reinforcement Learning: An Introduction*. MIT Press (2018).